In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_reservas AS
SELECT 
    hotel,
    is_canceled,
    lead_time,
    arrival_date_year,
    arrival_date_month,
    arrival_date_day_of_month,
    distribution_channel,
    reservation_status_date,
    adr,
    children,
    babies,
    total_of_special_requests,
    booking_changes,
    country
FROM cursodataacademy.proyecto_hotel.hotel_bookings;


1.(CREATE / INSERT) El equipo comercial quiere hacer un experimento rápido. Crea una tabla temporal (tb_reservas_prueba) y usa el comando INSERT con SELECT para insertar únicamente las reservas que corresponden a 'City Hotel' y que fueron canceladas (is_canceled = 1).

RPTA: La creación de esta tabla aislada permite auditar de forma focalizada las cancelaciones registradas en la categoría City Hotel. Al concentrar la mayor pérdida de volumen del hotel, esta segmentación facilita diseñar estrategias de retención específicas para el público urbano sin afectar la base de datos principal.

In [0]:
CREATE OR REPLACE TEMP TABLE tb_reservas_prueba AS 
SELECT * 
FROM cursodataacademy.proyecto_hotel.tb_reservas 
WHERE hotel = 'City Hotel' AND is_canceled = 1;

2. (UPDATE) Se detectó un error en los sistemas de captura de datos: los huéspedes que registraron babies en realidad no tenían bebés, sino que era un error del canal. Actualiza la tabla tb_reservas_prueba y cambia el valor de babies a 0 para todos los registros.

RPTA: La corrección del error de captura eliminó distorsiones operativas en los 33,102 registros de City Hotel. Ajustar el dato de babies a cero previene la sobreestimación de demanda de cunas e infraestructura infantil, optimizando el aprovisionamiento de inventario real en el hotel.

In [0]:
UPDATE tb_reservas_prueba
SET babies = 0;

3.(CAST / CONCAT) Necesitamos tener la fecha de llegada en un formato legible. Selecciona las columnas de año, mes (que viene en texto, ej. 'July') y día, y concaténalas en una nueva columna llamada arrival_date_string con el formato "DD-Mes-YYYY" (Ej: "25-July-2015").

RPTA: La estandarización de los componentes temporales en una sola cadena (DD-Mes-YYYY) facilita la lectura ejecutiva de los itinerarios. Esto optimiza el cruce de información con sistemas externos de recepción y agiliza la reportería operativa de llegadas al hotel.

In [0]:
SELECT 
  arrival_date_day_of_month,
  arrival_date_month,
  arrival_date_year,
  CONCAT(CAST(arrival_date_day_of_month AS STRING),'-',arrival_date_month,'-',CAST(arrival_date_year AS STRING)) AS arrival_date_string
FROM cursodataacademy.proyecto_hotel.tb_reservas;

4.(LIKE) Queremos filtrar a un segmento muy específico del canal de distribución. Selecciona todas las reservas donde distribution_channel empiece con la letra 'T'.

RPTA: El filtrado con el patrón 'T%' aísla el comportamiento de los canales impulsados por intermediarios (como TA/TO - Tour Operators y Travel Agents). Esto permite medir la dependencia comercial hacia agencias de viajes intermediarias y ajustar las comisiones frente a la venta directa.


In [0]:
SELECT *
FROM cursodataacademy.proyecto_hotel.tb_reservas
WHERE distribution_channel LIKE 'T%';

5.(COUNT / SUM / AVG) Calcula a nivel global para todo el hotel: la cantidad total de reservas, la suma total de solicitudes especiales (total_of_special_requests) y el promedio del precio por noche (adr).

RPTA: El cálculo de los kpi globales consolida el volumen total de operaciones, la carga total de requerimientos operativos y la tarifa promedio por noche ($101.83). Estos indicadores resumen la escala macro del negocio y sirven como línea base (baseline) para proyectar ingresos globales e infraestructura de servicio al cliente

In [0]:
SELECT 
  COUNT(*) AS total_reservas,
  SUM(total_of_special_requests) AS suma_total_solicitudes_especiales,
  AVG(adr) AS promedio_precio_noche
FROM cursodataacademy.proyecto_hotel.tb_reservas;

6.(DELETE) En tu tabla de pruebas (tb_reservas_prueba), elimina todas las reservas que tuvieron un precio por noche (adr) igual a 0.0, ya que se consideran pruebas del sistema y no reservas reales.

RPTA: La eliminación de 129 registros con tarifa cero depura la tabla de pruebas, descartando fallos esto asegura que las métricas y los análisis posteriores reflejen transacciones comerciales reales y con valor monetario efectivo.

In [0]:
DELETE FROM tb_reservas_prueba
WHERE adr = 0.0;

7. (GROUP BY) El gerente quiere saber cuántas reservas entraron por cada canal de distribución. Obtén la cantidad de reservas agrupadas por distribution_channel.

RPTA: El análisis revela una alta dependencia hacia los operadores turísticos y agencias de viajes (TA/TO), que concentran 97,870 reservas del total. Diversificar la captación fortaleciendo los canales directos (14,645) permitirá reducir las comisiones a terceros e incrementar la rentabilidad nula por habitación.


In [0]:
SELECT 
  distribution_channel,
  COUNT(*) AS cantidad_reservas
FROM cursodataacademy.proyecto_hotel.tb_reservas
GROUP BY distribution_channel
ORDER BY cantidad_reservas DESC;

8.(CASE WHEN / Tipos de Datos) Crea una consulta que clasifique las reservas en base al precio (adr): Si es menor a 50 es "Económico", si está entre 50 y 150 es "Estándar", y mayor a 150 es "Premium".

RPTA: La segmentación del adr en categorías de precio permite identificar la distribución del inventario según el poder adquisitivo del cliente. Esta estratificación facilita el diseño de promociones dirigidas en los segmentos económico" y estándar.

In [0]:
SELECT 
  hotel,
  adr,
  CASE 
    WHEN adr < 50 THEN 'Económico'
    WHEN adr BETWEEN 50 AND 150 THEN 'Estándar'
    ELSE 'Premium'
  END AS categoria_precio
FROM cursodataacademy.proyecto_hotel.tb_reservas;

9.(Funciones de Fecha - DATE_SUB / DATE_ADD) (Considera que existe una columna llamada arrival_date de tipo DATE para este ejercicio). Sabemos el lead_time (días de anticipación). Usa la función de Databricks para restar los días de lead_time a la arrival_date y así obtener la fecha exacta en la que el cliente hizo la reservación. Llámala booking_date.

RPTA: El cálculo retroactivo de la fecha exacta de reserva (booking_date) mediante el lead_time permite auditar con precisión los plazos de planificación del cliente. Esto facilita el diseño de campañas de marketing anticipadas y la optimización de políticas de cancelación.

In [0]:
SELECT 
  to_date(CONCAT(arrival_date_year, '-', arrival_date_month, '-', arrival_date_day_of_month), 'yyyy-MMMM-d') AS arrival_date,lead_time,DATE_SUB(to_date(CONCAT(arrival_date_year, '-', arrival_date_month, '-', arrival_date_day_of_month), 'yyyy-MMMM-d'),CAST(lead_time AS INT)
  ) AS booking_date
FROM cursodataacademy.proyecto_hotel.tb_reservas;

10.(Funciones Matemáticas) Queremos calcular una "cuota de riesgo". Selecciona el lead_time, divídelo entre 30 (para obtener meses aproximados) y usa la función CEIL() (redondeo hacia arriba) para mostrar el resultado en una columna llamada meses_anticipacion_techo.

RPTA: Transformar el tiempo de anticipación (lead_time) en bloques mensuales mediante el redondeo superior (CEIL) permite estratificar temporalmente el riesgo de cancelación. Esto facilita el diseño de políticas de penalización y plazos de pago escalonados según los meses previos a la llegada del huésped.

In [0]:
SELECT 
  lead_time,
  CEIL(lead_time / 30.0) AS meses_anticipacion_techo
FROM cursodataacademy.proyecto_hotel.tb_reservas;